In [2]:
from sqlalchemy import Boolean, Column, DateTime, ForeignKey, Integer, String, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import relationship
from sqlalchemy.sql import func
from sqlalchemy.orm import relationship
from sqlalchemy.orm.decl_api import DeclarativeBase

class Base(DeclarativeBase):
    pass


class User(Base):
    """User account."""

    __tablename__ = "user"

    id = Column(Integer, primary_key=True, autoincrement="auto")
    username = Column(String(255), unique=True, nullable=False)
    password = Column(Text, nullable=False)
    email = Column(String(255), unique=True, nullable=False)
    first_name = Column(String(255))
    last_name = Column(String(255))
    bio = Column(Text)
    avatar_url = Column(Text)
    role = Column(String(255))
    last_seen = Column(DateTime)
    created_at = Column(DateTime, server_default=func.now())
    updated_at = Column(DateTime, onupdate=func.now())

    def __repr__(self):
        return f"<User {self.username}>"


class Comment(Base):
    """User-generated comment on a blog post."""

    __tablename__ = "comment"

    id = Column(Integer, primary_key=True, index=True)
    user_id = Column(Integer, ForeignKey("user.id"))  # FK added
    post_id = Column(Integer, ForeignKey("post.id"), index=True)  # FK added
    body = Column(Text)
    upvotes = Column(Integer, default=1)
    removed = Column(Boolean, default=False)
    created_at = Column(DateTime, server_default=func.now())

    # Relationships
    user = relationship("User")

    def __repr__(self):
        return f"<Comment {self.id}>"


class Post(Base):
    """Blog post/article."""

    __tablename__ = "post"

    id = Column(Integer, primary_key=True, index=True)
    author_id = Column(Integer, ForeignKey("user.id"))  # FK added
    slug = Column(String(255), nullable=False, unique=True)
    title = Column(String(255), nullable=False)
    summary = Column(String(400))
    feature_image = Column(String(300))
    body = Column(Text)
    status = Column(String(255), nullable=False, default="unpublished")
    created_at = Column(DateTime, server_default=func.now())
    updated_at = Column(DateTime, server_default=func.now())
    author_id = Column(Integer, ForeignKey("user.id"))
    # Relationships
    author = relationship("User")
    comments = relationship("Comment")

    def __repr__(self):
        return f"<Post {self.id}>"

In [5]:
from sqlalchemy import Column, DateTime, String, Integer, ForeignKey, func
from sqlalchemy.orm import relationship, backref
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

Base = declarative_base()
 
class Person(Base):
    __tablename__ = 'person'
    # Here we define columns for the table person
    # Notice that each column is also a normal Python instance attribute.
    id = Column(Integer, primary_key=True)
    name = Column(String(250), nullable=False)
 
class Address(Base):
    __tablename__ = 'address'
    # Here we define columns for the table address.
    # Notice that each column is also a normal Python instance attribute.
    id = Column(Integer, primary_key=True)
    street_name = Column(String(250))
    street_number = Column(String(250))
    post_code = Column(String(250), nullable=False)
    person_id = Column(Integer, ForeignKey('person.id'))
    person = relationship(Person)
 
# Create an engine that stores data in the local directory's
# sqlalchemy_example.db file.
engine = create_engine('sqlite:///exemplo.db')
 
# Create all tables in the engine. This is equivalent to "Create Table"
# statements in raw SQL.
Base.metadata.create_all(engine)

C:\Users\z004n22k\AppData\Local\Temp\ipykernel_24196\1744676016.py:7: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [6]:
SessionLocal = sessionmaker(
    expire_on_commit=False, autocommit=False, autoflush=False, bind=engine
)
db = SessionLocal()


In [15]:
address = Address(
    street_name = "SM NAGAR",
    street_number = "12",
    post_code = "266",
    person_id = 1
    )

db.add_all([address])

In [16]:
db.query(Address).all()

[]

ModuleNotFoundError: No module named 'sqlalchemy.interfaces'

In [3]:
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, ForeignKeyConstraint, event
from sqlalchemy.orm import sessionmaker, relationship
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.exc import IntegrityError

Base = declarative_base()

class Person(Base):
    __tablename__ = "persons"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False, unique=True)

    posts = relationship("Post", back_populates="author")

class Post(Base):
    __tablename__ = "posts"

    id = Column(Integer, primary_key=True, index=True)
    title = Column(String, nullable=False)
    content = Column(String, nullable=False)

    author_id = Column(Integer, ForeignKey("persons.id", ondelete="CASCADE", onupdate="CASCADE"), nullable=False)
    author = relationship("Person", back_populates="posts")

    __table_args__ = (
        ForeignKeyConstraint(['author_id'], ['persons.id'], name='author_fk'),
    )


# Create the SQLite database and tables
engine = create_engine("sqlite:///mydatabase.db", echo=True)

@event.listens_for(engine, "connect")
def enable_foreign_keys(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.close()
    
Base.metadata.create_all(engine)


# Create a session to interact with the database
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
db = SessionLocal()


2023-10-08 17:53:17,034 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2023-10-08 17:53:17,038 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("persons")
2023-10-08 17:53:17,041 INFO sqlalchemy.engine.Engine [raw sql] ()
2023-10-08 17:53:17,045 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("persons")
2023-10-08 17:53:17,048 INFO sqlalchemy.engine.Engine [raw sql] ()
2023-10-08 17:53:17,051 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("posts")
2023-10-08 17:53:17,053 INFO sqlalchemy.engine.Engine [raw sql] ()
2023-10-08 17:53:17,056 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("posts")
2023-10-08 17:53:17,059 INFO sqlalchemy.engine.Engine [raw sql] ()
2023-10-08 17:53:17,065 INFO sqlalchemy.engine.Engine 
CREATE TABLE persons (
	id INTEGER NOT NULL, 
	name VARCHAR NOT NULL, 
	PRIMARY KEY (id), 
	UNIQUE (name)
)


2023-10-08 17:53:17,069 INFO sqlalchemy.engine.Engine [no key 0.00422s] ()
2023-10-08 17:53:17,082 INFO sqlalchemy.engine.Engine CREATE INDEX ix_per

2023-10-08 17:53:17,098 INFO sqlalchemy.engine.Engine 
CREATE TABLE posts (
	id INTEGER NOT NULL, 
	title VARCHAR NOT NULL, 
	content VARCHAR NOT NULL, 
	author_id INTEGER NOT NULL, 
	PRIMARY KEY (id), 
	CONSTRAINT author_fk FOREIGN KEY(author_id) REFERENCES persons (id), 
	FOREIGN KEY(author_id) REFERENCES persons (id) ON DELETE CASCADE ON UPDATE CASCADE
)


2023-10-08 17:53:17,103 INFO sqlalchemy.engine.Engine [no key 0.00520s] ()
2023-10-08 17:53:17,123 INFO sqlalchemy.engine.Engine CREATE INDEX ix_posts_id ON posts (id)
2023-10-08 17:53:17,126 INFO sqlalchemy.engine.Engine [no key 0.00334s] ()
2023-10-08 17:53:17,139 INFO sqlalchemy.engine.Engine COMMIT


C:\Users\z004n22k\AppData\Local\Temp\ipykernel_24196\2255744790.py:6: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [6]:
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
db = SessionLocal()

# Example usage: Adding a post with a non-existing author
try:
    new_post = Post(title="Example Post", content="This is an example post.", author_id=1)
    db.add(new_post)
    db.commit()
except IntegrityError:
    db.rollback()
    print("Author does not exist. Please create the author first.")

# Close the session when you're done
db.close()

2023-10-08 17:53:35,831 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2023-10-08 17:53:35,836 INFO sqlalchemy.engine.Engine INSERT INTO posts (title, content, author_id) VALUES (?, ?, ?)
2023-10-08 17:53:35,840 INFO sqlalchemy.engine.Engine [cached since 14.1s ago] ('Example Post', 'This is an example post.', 1)
2023-10-08 17:53:35,846 INFO sqlalchemy.engine.Engine ROLLBACK
Author does not exist. Please create the author first.


In [4]:
db.query(Post).all()

2023-10-08 17:29:06,782 INFO sqlalchemy.engine.Engine SELECT posts.id AS posts_id, posts.title AS posts_title, posts.content AS posts_content, posts.author_id AS posts_author_id 
FROM posts
2023-10-08 17:29:06,786 INFO sqlalchemy.engine.Engine [generated in 0.00250s] ()


In [1]:
import motor.motor_asyncio
from urllib.parse import quote_plus

MONGO_DETAILS = "mongodb+srv://bharath0292:" + quote_plus("Findrey@123")+"@personal.23ipegd.mongodb.net/?retryWrites=true&w=majority"

client = motor.motor_asyncio.AsyncIOMotorClient(MONGO_DETAILS)

database = client.FinDrey

In [19]:
collection = database.get_collection("TransactionCounter")

In [16]:
collection.insert_one({
   "name":"Jhon"
})


<Future pending cb=[_chain_future.<locals>._call_check_cancel() at c:\Users\z004n22k\AppData\Local\anaconda3\envs\quantdrey\Lib\asyncio\futures.py:387]>

In [20]:
async for student in collection.find({}):
    print(student)

{'_id': {'db': 'FinDrey', 'coll': 'Transactions'}, 'seq_value': 3}
{'_id': ObjectId('6532a66180a6553a5a455451'), 'name': 'Jhon'}


In [50]:
transaction_type = []

async for student in collection.find({}):
    print(student)
    transaction_type.append(student["account_type"])

{'_id': ObjectId('652423271bd43b5619c343c6'), 'account_type': 'INHAND'}
{'_id': ObjectId('6524234e1bd43b5619c343c7'), 'account_type': 'BANK ACCOUNT'}
{'_id': ObjectId('652423681bd43b5619c343c8'), 'account_type': 'CREDIT CARD'}
{'_id': ObjectId('652423771bd43b5619c343c9'), 'account_type': 'WALLET'}
{'_id': ObjectId('65311ff0e1c09122c96be061'), 'account_type': 'LOAN A/C'}
{'_id': ObjectId('65311ffee1c09122c96be062'), 'account_type': 'INVESTMENT A/C'}
{'_id': ObjectId('6531201ae1c09122c96be063'), 'account_type': 'RECURRING DEPOSIT'}
{'_id': ObjectId('65312029e1c09122c96be064'), 'account_type': 'FIXED DEPOSIT'}


In [28]:
async def add_student(student_data: dict) -> dict:
    student = await collection.insert_one(student_data)
    new_student = await collection.find_one({"_id": student.inserted_id})
    return student_helper(new_student)

In [38]:
from bson import  ObjectId

In [44]:
collection.insert_one({'_id': 'icici', 'new': 'icici'})

<Future pending cb=[_chain_future.<locals>._call_check_cancel() at c:\Users\z004n22k\AppData\Local\anaconda3\envs\quantdrey\Lib\asyncio\futures.py:387]>

In [35]:
from enum import StrEnum
Animal = StrEnum('Animal', transaction_type, module=__name__)

Animal['Cash Recieved']

<Animal.Cash Recieved: 'cash recieved'>

In [51]:
transaction_type

['INHAND',
 'BANK ACCOUNT',
 'CREDIT CARD',
 'WALLET',
 'LOAN A/C',
 'INVESTMENT A/C',
 'RECURRING DEPOSIT',
 'FIXED DEPOSIT']

In [26]:
from enum import StrEnum

class TransactionType(StrEnum):
    EXPENSE= "Expense"
    ATM_Withdrawal= "ATM Withdrawal"
    Cash_Recieved= "Cash Recieved"
    Cashback= "Cashback"
    Refund= "Refund"
    Lend= "Lend"
    Transfer= "Transfer"
    Income= "Income"
    Debt= "Debt"


TransactionType.EXPENSE


<TransactionType.EXPENSE: 'Expense'>

In [2]:
import json

with open('../common-enums.json', 'r') as file:
    enums = json.load(file)
    print(enums['Status'])

{'ACTIVE': 'active', 'INACTIVE': 'inactive'}
